In [ ]:
## Transformer les JSON extraits en format unique exploitable : csv

In [1]:
import pandas as pd
import json
import os

In [5]:
for path in paths:
    for file in os.listdir(path):
        if file.endswith(".json"):
            with open(os.path.join(path, file), encoding="utf-8") as f:
                data = json.load(f)
                all_data.extend(data)

In [6]:
df = pd.json_normalize(all_data)


In [2]:
output_path = r"D:\Mes projets\PFE\Project\data\processed\dataset.csv"
df.to_csv(output_path, index=False, encoding="utf-8")

print("CSV généré :", output_path)

CSV généré : D:\Mes projets\PFE\Project\data\processed\dataset.csv


In [3]:
# D'abord, vérifiez les noms de colonnes existants
print(df.columns.tolist())

['refcode', 'doi', 'mof.nom', 'mof.famille', 'mof.metal', 'mof.etat_oxydation', 'mof.ligand', 'mof.groupe_fonctionnel', 'mof.sites_metalliques_ouverts', 'mof.type_structure', 'cocatalyseur.type', 'cocatalyseur.charge_pct', 'conditions_reaction.source_lumiere', 'conditions_reaction.longueur_onde_nm', 'conditions_reaction.intensite_mW_cm2', 'conditions_reaction.agent_sacrificiel', 'conditions_reaction.solvant', 'conditions_reaction.concentration_catalyseur_g_L', 'conditions_reaction.pression_CO2_bar', 'conditions_reaction.temperature_C', 'conditions_reaction.duree_irradiation_h', 'conditions_reaction.atmosphere', 'proprietes_structurales.BET_m2_g', 'proprietes_structurales.volume_pores_cm3_g', 'proprietes_structurales.PLD_angstrom', 'proprietes_structurales.void_fraction', 'proprietes_electroniques.band_gap_eV', 'proprietes_electroniques.HOMO_eV_vs_NHE', 'proprietes_electroniques.LUMO_eV_vs_NHE', 'proprietes_electroniques.absorption_visible', 'performance.produit_majoritaire', 'performan

In [4]:
data = pd.read_csv('D:\Mes projets\PFE\Project\data\processed\dataset.csv')
# Check for missing values in each column
print(data.isnull().sum())
# Check the percentage of missing data per column
print(data.isnull().mean() * 100)

refcode                                                       0
doi                                                          11
mof.nom                                                       0
mof.famille                                                   0
mof.metal                                                     8
mof.etat_oxydation                                           30
mof.ligand                                                   16
mof.groupe_fonctionnel                                       24
mof.sites_metalliques_ouverts                               172
mof.type_structure                                            0
cocatalyseur.type                                             0
cocatalyseur.charge_pct                                     254
conditions_reaction.source_lumiere                            0
conditions_reaction.longueur_onde_nm                        164
conditions_reaction.intensite_mW_cm2                        262
conditions_reaction.agent_sacrificiel   

In [5]:
print("\n📊 Types de données :")
print(df.dtypes)


📊 Types de données :
refcode                                                         str
doi                                                             str
mof.nom                                                         str
mof.famille                                                     str
mof.metal                                                       str
mof.etat_oxydation                                              str
mof.ligand                                                      str
mof.groupe_fonctionnel                                          str
mof.sites_metalliques_ouverts                                object
mof.type_structure                                              str
cocatalyseur.type                                               str
cocatalyseur.charge_pct                                     float64
conditions_reaction.source_lumiere                              str
conditions_reaction.longueur_onde_nm                        float64
conditions_reaction.intens

In [10]:
# =============================================================
# 0. IMPORTS & GLOBAL STYLE  (paper-ready, compact, no bold)
# =============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path

# --- Paper style ---
mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.titleweight": "normal",
    "axes.labelweight": "normal",
    "font.weight": "normal",
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
sns.set_style("whitegrid")
sns.set_context("paper")

OUT = Path(r"D:\Mes projets\PFE\fastapi-ml\data\processed\figures")
OUT.mkdir(parents=True, exist_ok=True)

# --- Load ---
df = pd.read_csv(r"D:\Mes projets\PFE\fastapi-ml\data\processed\dataset.csv")
print("Shape:", df.shape)
print("Unique articles (doi):", df["doi"].nunique())
print("Unique refcodes:", df["refcode"].nunique())

# Categorical variables to collapse rare labels
def collapse(series, top_n=8, other="Other"):
    s = series.astype("string").str.strip()
    s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "N/A": pd.NA})
    counts = s.value_counts(dropna=True)
    keep = counts.head(top_n).index
    return s.where(s.isin(keep), other).fillna("Not reported")

def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(OUT / f"{name}.{ext}")
    plt.close(fig)
    print("saved:", name)

Shape: (316, 48)
Unique articles (doi): 83
Unique refcodes: 158


In [31]:
# =============================================================
# FIG 1 — Missing-value rate per variable (before preprocessing)
#   - non-variable columns removed
#   - near-empty columns removed
#   - threshold 60% (excluded vars shown hatched, not hidden)
#   - legend outside, right
#   - exact count in title
# =============================================================
import matplotlib.patches as mpatches

# ---------- 1) Colonnes exclues ----------
DROP_NONVARS = [
    "doi", "refcode", "mof.nom",
    "qualite.notes", "qualite.features_manquantes",
    "proprietes_structurales.notes",
]
df_v = df.drop(columns=[c for c in DROP_NONVARS if c in df.columns])

DROP_EMPTY = [
    "proprietes_electroniques.band_gap_ev",
    "proprietes_structurales.PLD_nm",
]
df_v = df_v.drop(columns=[c for c in DROP_EMPTY if c in df_v.columns])

# ---------- 2) Mapping labels courts + catégories ----------
VAR_MAP = {
    "mof.famille":                          ("MOF family",          "MOF"),
    "mof.type_structure":                   ("Structure type",      "MOF"),
    "mof.metal":                            ("Metal",               "MOF"),
    "mof.ligand":                           ("Ligand",              "MOF"),
    "mof.groupe_fonctionnel":               ("Functional group",    "MOF"),
    "mof.etat_oxydation":                   ("Oxidation state",     "MOF"),
    "mof.sites_metalliques_ouverts":        ("Open metal sites",    "MOF"),
    "proprietes_structurales.BET_m2_g":     ("BET area",            "Structure"),
    "proprietes_structurales.volume_pores_cm3_g": ("Pore volume",   "Structure"),
    "proprietes_structurales.void_fraction":("Void fraction",      "Structure"),
    "proprietes_structurales.PLD_angstrom": ("PLD",                 "Structure"),
    "proprietes_electroniques.band_gap_eV": ("Band gap",            "Electronic"),
    "proprietes_electroniques.HOMO_eV_vs_NHE": ("HOMO",             "Electronic"),
    "proprietes_electroniques.LUMO_eV_vs_NHE": ("LUMO",             "Electronic"),
    "proprietes_electroniques.absorption_visible": ("Visible absorption", "Electronic"),
    "conditions_reaction.source_lumiere":   ("Light source",        "Conditions"),
    "conditions_reaction.agent_sacrificiel":("Sacrificial agent",   "Conditions"),
    "conditions_reaction.solvant":          ("Solvent",             "Conditions"),
    "conditions_reaction.atmosphere":       ("Atmosphere",          "Conditions"),
    "conditions_reaction.pression_CO2_bar": ("pCO2",                "Conditions"),
    "conditions_reaction.pression_O2_bar":  ("pO2",                 "Conditions"),
    "conditions_reaction.temperature_C":    ("Temperature",         "Conditions"),
    "conditions_reaction.duree_irradiation_h": ("Irradiation time", "Conditions"),
    "conditions_reaction.longueur_onde_nm": ("Wavelength",          "Conditions"),
    "conditions_reaction.intensite_mW_cm2": ("Light intensity",     "Conditions"),
    "conditions_reaction.concentration_catalyseur_g_L": ("Catalyst conc.", "Conditions"),
    "cocatalyseur.type":                    ("Co-catalyst",         "Conditions"),
    "cocatalyseur.charge_pct":              ("Co-cat. loading",     "Conditions"),
    "performance.produit_majoritaire":      ("Major product",       "Performance"),
    "performance.type_taux":                ("Rate type",           "Performance"),
    "performance.taux_production_umol_g_h": ("Production rate",     "Target"),
    "performance.selectivite_pct":          ("Selectivity",         "Performance"),
    "performance.AQE_pct":                  ("AQE",                 "Performance"),
    "performance.TON":                      ("TON",                 "Performance"),
    "performance.H2_concurrent_umol_g_h":   ("H2 co-production",    "Performance"),
    "performance_photocatalytique.co_production_rate_umol_g_h": ("CO co-production", "Performance"),
    "stabilite_photocatalytique.nombre_cycles": ("Cycles",          "Stability"),
    "stabilite_photocatalytique.duree_totale_h": ("Total duration", "Stability"),
    "stabilite_photocatalytique.retention_pct": ("Retention",       "Stability"),
    "stabilite_photocatalytique.caracterisation_post": ("Post-characterization", "Stability"),
}

# ---------- 3) Calcul du taux de manquant sur TOUTES les variables mappées ----------
present = [c for c in VAR_MAP if c in df_v.columns]
miss = df_v[present].isna().mean()

rows = pd.DataFrame({
    "col":      present,
    "label":    [VAR_MAP[c][0] for c in present],
    "cat":      [VAR_MAP[c][1] for c in present],
    "miss_pct": (miss[present].values * 100),
})

# ---------- 4) Seuil 60 % ----------
THRESH = 60
rows["kept"] = rows["miss_pct"] <= THRESH
n_kept   = int(rows["kept"].sum())
n_total  = len(rows)
n_excl   = n_total - n_kept

# ---------- 5) Tri : gardées d'abord (tri intra par catégorie), exclues en bas ----------
CAT_ORDER = ["MOF", "Structure", "Electronic", "Conditions",
             "Performance", "Target", "Stability"]
CAT_RANK  = {c: i for i, c in enumerate(CAT_ORDER)}
rows["cat_rank"] = rows["cat"].map(CAT_RANK)

kept    = rows[rows["kept"]].sort_values(["cat_rank", "miss_pct"],
                                         ascending=[True, True])
excl    = rows[~rows["kept"]].sort_values("miss_pct", ascending=True)
rows_pl = pd.concat([kept, excl]).reset_index(drop=True)

# ---------- 6) Couleurs ----------
COLORS = {
    "MOF":         "#4C72B0",
    "Structure":   "#55A868",
    "Electronic":  "#8172B2",
    "Conditions":  "#DD8452",
    "Performance": "#C44E52",
    "Target":      "#000000",
    "Stability":   "#CCB974",
}

# ---------- 7) Figure ----------
fig, ax = plt.subplots(figsize=(5.6, 7.2))
y = np.arange(len(rows_pl))
colors = rows_pl["cat"].map(COLORS).values

# barres : gardées = pleines ; exclues = hachurées + grises (mais couleur conservée en bord)
for i, (_, r) in enumerate(rows_pl.iterrows()):
    if r["kept"]:
        ax.barh(i, r["miss_pct"], color=COLORS[r["cat"]],
                edgecolor="none", height=0.78)
    else:
        ax.barh(i, r["miss_pct"], color=COLORS[r["cat"]],
                edgecolor="none", height=0.78, alpha=0.25)
        ax.barh(i, r["miss_pct"], color="none",
                edgecolor="dimgrey", height=0.78, hatch="////", lw=0.4)

ax.set_yticks(y)
ax.set_yticklabels(rows_pl["label"], fontsize=7)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Missing values (%)")
ax.set_title(
    f"Missing-value rate per variable (n = {len(df)})\n"
    f"{n_kept}/{n_total} variables retained "
    f"(threshold {THRESH} %, {n_excl} excluded)",
    loc="left", fontsize=9,
)

# seuil
ax.axvline(THRESH, color="grey", lw=0.6, ls="--")
ax.text(THRESH + 0.7, len(rows_pl) - 0.3, f"{THRESH} %",
        fontsize=6, color="grey", va="bottom")

# valeurs en bout de barre
for i, r in enumerate(rows_pl.itertuples()):
    txt = f"{r.miss_pct:.0f}"
    ax.text(r.miss_pct + 1, i, txt, va="center", fontsize=5.5,
            color=("black" if r.cat == "Target" else
                   ("dimgrey" if not r.kept else "black")))

for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)

# ---------- 8) Légende HORS du graphique (à droite) ----------
handles = [mpatches.Patch(color=COLORS[k], label=k)
           for k in CAT_ORDER if k in set(rows_pl["cat"])]
handles += [
    mpatches.Patch(facecolor="lightgrey", edgecolor="dimgrey",
                   hatch="////", label=f"Excluded (>{THRESH} %)"),
]
ax.legend(handles=handles, frameon=False,
          loc="center left", bbox_to_anchor=(1.02, 0.5),
          fontsize=6.5, title="Variable group",
          title_fontsize=7, borderaxespad=0.)

fig.tight_layout()
save(fig, "fig01_missingness")

saved: fig01_missingness


In [26]:
# =============================================================
# 0bis. collapse() corrigée — "autre"/"Autre" -> "Other"
# =============================================================
def collapse(series, top_n=8, other="Other"):
    s = series.astype("string").str.strip()
    s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "N/A": pd.NA})
    # Normalisation FR -> EN (accents + casse)
    s = s.str.replace(r"(?i)^autre$", "Other", regex=True)
    s = s.str.replace(r"(?i)^others?$", "Other", regex=True)
    s = s.str.replace(r"(?i)^non\s*renseign[ée]$", "Not reported", regex=True)
    s = s.str.replace(r"(?i)^non\s*report[ée]?$", "Not reported", regex=True)

    counts = s.value_counts(dropna=True)
    keep = counts.head(top_n).index
    out = s.where(s.isin(keep), other).fillna("Not reported")
    # Si "Other" apparait dans le top, on le garde tel quel (pas de doublon)
    return out
# =============================================================
# FIG 2 — MOF family distribution  (MOF-derived -> Other)
# =============================================================
fam_raw = (df["mof.famille"]
             .astype("string").str.strip()
             .str.replace(r"(?i).*mof[\s\-_/]*derived.*", "Other", regex=True))

fam = collapse(fam_raw, top_n=8)          

fig, ax = plt.subplots(figsize=(3.4, 2.6))
order = fam.value_counts().index
sns.countplot(y=fam, order=order, ax=ax, color="#4C72B0", edgecolor="none")
ax.set_xlabel("Number of entries")
ax.set_ylabel("")
ax.set_title(f"MOF families (n = {len(df)})", loc="left")
for i, v in enumerate(fam.value_counts().reindex(order).values):
    ax.text(v + 0.3, i, str(v), va="center", fontsize=7)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
save(fig, "fig02_mof_families")

# =============================================================
# FIG 3 — Metal distribution
# =============================================================
metal = collapse(df["mof.metal"], top_n=10)

fig, ax = plt.subplots(figsize=(3.4, 2.8))
order = metal.value_counts().index
sns.countplot(y=metal, order=order, ax=ax, color="#55A868", edgecolor="none")
ax.set_xlabel("Number of entries")
ax.set_ylabel("")
ax.set_title(f"Metal nodes (n = {len(df)})", loc="left")
for i, v in enumerate(metal.value_counts().reindex(order).values):
    ax.text(v + 0.3, i, str(v), va="center", fontsize=7)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
save(fig, "fig03_metals")

# =============================================================
# FIG 4 — Ligand distribution
# =============================================================
lig = collapse(df["mof.ligand"], top_n=10)

fig, ax = plt.subplots(figsize=(3.4, 2.8))
order = lig.value_counts().index
sns.countplot(y=lig, order=order, ax=ax, color="#C44E52", edgecolor="none")
ax.set_xlabel("Number of entries")
ax.set_ylabel("")
ax.set_title(f"Ligands (n = {len(df)})", loc="left")
for i, v in enumerate(lig.value_counts().reindex(order).values):
    ax.text(v + 0.3, i, str(v), va="center", fontsize=7)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
save(fig, "fig04_ligands")

# =============================================================
# FIG 5 — Functional groups + oxidation states (2-panel)
# =============================================================
fg = collapse(df["mof.groupe_fonctionnel"], top_n=6)
ox = collapse(df["mof.etat_oxydation"], top_n=5)

fig, axes = plt.subplots(1, 2, figsize=(6.0, 2.6))
for ax, s, title, col in zip(
    axes,
    [fg, ox],
    ["Functional groups", "Metal oxidation states"],
    ["#8172B2", "#CCB974"],
):
    order = s.value_counts().index
    sns.countplot(y=s, order=order, ax=ax, color=col, edgecolor="none")
    ax.set_xlabel("Number of entries")
    ax.set_ylabel("")
    ax.set_title(title, loc="left")
    for i, v in enumerate(s.value_counts().reindex(order).values):
        ax.text(v + 0.2, i, str(v), va="center", fontsize=6)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
fig.tight_layout()
save(fig, "fig05_functional_groups_oxidation")
# =============================================================
# FIG 6 — Composite 2x2 panel (a) family (b) structure type
#                       (c) ligand (d) metal
# Compact, half-page, labels a) b) c) d)
# =============================================================
fam_c = collapse(fam_raw, top_n=8) 
struct  = collapse(df["mof.type_structure"], top_n=8)
lig_c   = collapse(df["mof.ligand"],       top_n=8)
met_c   = collapse(df["mof.metal"],        top_n=8)

panels = [
    ("a)", fam_c,   "MOF family",        "#4C72B0"),
    ("b)", struct,  "Structure type",    "#DD8452"),
    ("c)", lig_c,   "Ligand",            "#C44E52"),
    ("d)", met_c,   "Metal node",        "#55A868"),
]

fig, axes = plt.subplots(2, 2, figsize=(7.0, 5.2))
for ax, (tag, s, title, col) in zip(axes.ravel(), panels):
    order = s.value_counts().index
    sns.countplot(y=s, order=order, ax=ax, color=col, edgecolor="none")
    ax.set_xlabel("Number of entries")
    ax.set_ylabel("")
    # Titre = label + nom du panneau (normal, pas gras)
    ax.set_title(f"{tag} {title}", loc="left", fontsize=9)
    # valeurs en bout de barre
    for i, v in enumerate(s.value_counts().reindex(order).values):
        ax.text(v + 0.25, i, str(v), va="center", fontsize=6)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

fig.suptitle(f"Composition of the curated dataset (n = {len(df)})",
             x=0.01, ha="left", fontsize=10)
fig.tight_layout(rect=[0, 0, 1, 0.96])
save(fig, "fig06_composite_family_structure_ligand_metal")

saved: fig02_mof_families
saved: fig03_metals
saved: fig04_ligands
saved: fig05_functional_groups_oxidation
saved: fig06_composite_family_structure_ligand_metal


In [18]:
# =============================================================
# FIG 7 — Co-catalyst / sacrificial agent / solvent
# =============================================================
cocat = collapse(df["cocatalyseur.type"], top_n=7)
sacr  = collapse(df["conditions_reaction.agent_sacrificiel"], top_n=6)
solv  = collapse(df["conditions_reaction.solvant"], top_n=8)

fig, axes = plt.subplots(1, 3, figsize=(8.0, 2.8))
for ax, s, title, col in zip(
    axes,
    [cocat, sacr, solv],
    ["Co-catalyst", "Sacrificial agent", "Solvent"],
    ["#4C72B0", "#DD8452", "#55A868"],
):
    order = s.value_counts().index
    sns.countplot(y=s, order=order, ax=ax, color=col, edgecolor="none")
    ax.set_xlabel("Entries")
    ax.set_ylabel("")
    ax.set_title(title, loc="left")
    for i, v in enumerate(s.value_counts().reindex(order).values):
        ax.text(v + 0.2, i, str(v), va="center", fontsize=6)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
fig.tight_layout()
save(fig, "fig07_cocat_sacr_solvent")

saved: fig07_cocat_sacr_solvent


In [19]:
# =============================================================
# FIG 8 — Log-distributions of key continuous variables
# =============================================================
num_cols = [
    "performance.taux_production_umol_g_h",
    "performance.selectivite_pct",
    "performance.AQE_pct",
    "proprietes_structurales.BET_m2_g",
    "proprietes_electroniques.band_gap_eV",
    "conditions_reaction.pression_CO2_bar",
    "conditions_reaction.temperature_C",
    "conditions_reaction.duree_irradiation_h",
]

fig, axes = plt.subplots(2, 4, figsize=(9.5, 4.0))
for ax, c in zip(axes.ravel(), num_cols):
    s = pd.to_numeric(df[c], errors="coerce").dropna()
    if s.empty:
        ax.set_visible(False); continue
    use_log = (s > 0).all() and s.max() / max(s.min(), 1e-9) > 100
    data = np.log10(s) if use_log else s
    ax.hist(data, bins=25, color="#4C72B0", edgecolor="white", linewidth=0.3)
    label = c.split(".")[-1].replace("_", " ")
    ax.set_title(("log10 " if use_log else "") + label, loc="left")
    ax.set_ylabel("Count")
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
fig.tight_layout()
save(fig, "fig08_numeric_distributions")

saved: fig08_numeric_distributions


In [20]:
# =============================================================
# FIG 9 — Metals × MOF families (heatmap)
# =============================================================
ct = pd.crosstab(collapse(df["mof.metal"], 8), collapse(df["mof.famille"], 6))

fig, ax = plt.subplots(figsize=(4.2, 3.0))
sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", cbar=False,
            linewidths=0.4, linecolor="white", ax=ax,
            annot_kws={"fontsize": 6})
ax.set_xlabel("MOF family"); ax.set_ylabel("Metal")
ax.set_title("Metal × family", loc="left")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
save(fig, "fig09_metal_family_heatmap")

# =============================================================
# FIG 10 — Co-catalyst presence vs product (stacked)
# =============================================================
tmp = pd.DataFrame({
    "product": prod,
    "cocat": df["cocatalyseur.type"].astype("string").str.strip().notna().map({True:"with co-cat", False:"no co-cat"})
})
ct2 = pd.crosstab(tmp["product"], tmp["cocat"])
ct2 = ct2.loc[ct2.sum(axis=1).sort_values(ascending=True).index]

fig, ax = plt.subplots(figsize=(4.0, 2.8))
ct2.plot(kind="barh", stacked=True, ax=ax, color=["#DD8452", "#4C72B0"], edgecolor="white", linewidth=0.4)
ax.set_xlabel("Number of entries"); ax.set_ylabel("")
ax.set_title("Product vs co-catalyst", loc="left")
ax.legend(title="", frameon=False, loc="lower right")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
fig.tight_layout()
save(fig, "fig10_product_cocatalyst")

saved: fig09_metal_family_heatmap
saved: fig10_product_cocatalyst


In [21]:
# =============================================================
# Summary table (export CSV + markdown) — to paste in SI
# =============================================================
def summarize(series, name):
    s = series.astype("string").str.strip()
    n_missing = s.isna().sum() + (s == "").sum()
    vc = s.replace("", pd.NA).dropna().value_counts()
    return pd.DataFrame({
        "variable": name,
        "n_total": len(series),
        "n_missing": n_missing,
        "n_unique": vc.size,
        "top_1": vc.index[0] if len(vc) else "",
        "top_1_n": vc.iloc[0] if len(vc) else 0,
        "top_2": vc.index[1] if len(vc) > 1 else "",
        "top_2_n": vc.iloc[1] if len(vc) > 1 else 0,
    }, index=[0])

cat_vars = [
    "mof.famille", "mof.metal", "mof.ligand", "mof.groupe_fonctionnel",
    "mof.etat_oxydation", "mof.sites_metalliques_ouverts", "mof.type_structure",
    "cocatalyseur.type", "conditions_reaction.source_lumiere",
    "conditions_reaction.agent_sacrificiel", "conditions_reaction.solvant",
    "conditions_reaction.atmosphere", "performance.produit_majoritaire",
    "performance.type_taux", "proprietes_electroniques.absorption_visible",
]
summary = pd.concat([summarize(df[v], v) for v in cat_vars], ignore_index=True)
summary.to_csv(OUT / "table01_categorical_summary.csv", index=False)
print(summary)

# Numeric summary
num_all = df.select_dtypes(include=[np.number]).columns
num_sum = df[num_all].describe().T
num_sum.to_csv(OUT / "table02_numeric_summary.csv")

                                       variable  n_total  n_missing  n_unique  \
0                                   mof.famille      316          0        11   
1                                     mof.metal      316          8        50   
2                                    mof.ligand      316         16        47   
3                        mof.groupe_fonctionnel      316         24        16   
4                            mof.etat_oxydation      316         30         5   
5                 mof.sites_metalliques_ouverts      316        172         2   
6                            mof.type_structure      316          0         4   
7                             cocatalyseur.type      316          0        23   
8            conditions_reaction.source_lumiere      316          0        13   
9         conditions_reaction.agent_sacrificiel      316          0        13   
10                  conditions_reaction.solvant      316          0         9   
11               conditions_

In [27]:

fam_raw = (df["mof.famille"]
             .astype("string").str.strip()
             .str.replace(r"(?i).*mof[\s\-_/]*derived.*", "Other", regex=True))

fam_c  = collapse(fam_raw,                      top_n=8)
struct = collapse(df["mof.type_structure"],     top_n=8)
met_c  = collapse(df["mof.metal"],              top_n=8)

# heatmap : on réutilise les versions collapsed pour la cohérence
ct = pd.crosstab(met_c, fam_c)
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=False).index,
            ct.sum(axis=0).sort_values(ascending=False).index]

fig, axes = plt.subplots(2, 2, figsize=(7.0, 5.4))

# --- a) family ---
ax = axes[0, 0]
order = fam_c.value_counts().index
sns.countplot(y=fam_c, order=order, ax=ax, color="#4C72B0", edgecolor="none")
ax.set_xlabel("Number of entries"); ax.set_ylabel("")
ax.set_title("a) MOF family", loc="left", fontsize=9)
for i, v in enumerate(fam_c.value_counts().reindex(order).values):
    ax.text(v + 0.25, i, str(v), va="center", fontsize=6)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)

# --- b) structure type ---
ax = axes[0, 1]
order = struct.value_counts().index
sns.countplot(y=struct, order=order, ax=ax, color="#DD8452", edgecolor="none")
ax.set_xlabel("Number of entries"); ax.set_ylabel("")
ax.set_title("b) Structure type", loc="left", fontsize=9)
for i, v in enumerate(struct.value_counts().reindex(order).values):
    ax.text(v + 0.25, i, str(v), va="center", fontsize=6)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)

# --- c) metal ---
ax = axes[1, 0]
order = met_c.value_counts().index
sns.countplot(y=met_c, order=order, ax=ax, color="#55A868", edgecolor="none")
ax.set_xlabel("Number of entries"); ax.set_ylabel("")
ax.set_title("c) Metal node", loc="left", fontsize=9)
for i, v in enumerate(met_c.value_counts().reindex(order).values):
    ax.text(v + 0.25, i, str(v), va="center", fontsize=6)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)

# --- d) metal x family heatmap ---
ax = axes[1, 1]
sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", cbar=False,
            linewidths=0.4, linecolor="white", ax=ax,
            annot_kws={"fontsize": 6})
ax.set_xlabel("MOF family"); ax.set_ylabel("Metal")
ax.set_title("d) Metal × family", loc="left", fontsize=9)
plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
plt.setp(ax.get_yticklabels(), rotation=0)

fig.suptitle(f"Composition of the curated dataset (n = {len(df)})",
             x=0.01, ha="left", fontsize=10)
fig.tight_layout(rect=[0, 0, 1, 0.96])
save(fig, "fig02_composition")

saved: fig02_composition


In [28]:
# =============================================================
# FIG 3 — Conditions & target (4 panels):
#   (a) major product (b) rate type
#   (c) light source  (d) co-catalyst | sacrificial agent (combined)
# =============================================================
prod  = prodcat.map(map_product)                              # déjà défini
taux  = collapse(df["performance.type_taux"].map(map_rate_type), top_n=6)
light = collapse(df["conditions_reaction.source_lumiere"].map(map_light), top_n=8)

# panneau (d) combiné : co-catalyseur vs agent sacrificiel
cocat = df["cocatalyseur.type"].astype("string").str.strip()
cocat = cocat.where(~cocat.str.lower().isin(["none", "nan", "no", "n/a", ""]), "none")
cocat = collapse(cocat, top_n=6)

sacr = df["conditions_reaction.agent_sacrificiel"].astype("string").str.strip()
sacr = sacr.where(~sacr.str.lower().isin(["none", "nan", "no", "n/a", ""]), "none")
sacr = collapse(sacr, top_n=6)

fig, axes = plt.subplots(2, 2, figsize=(7.0, 5.2))

# --- a) major product ---
ax = axes[0, 0]
order = prod.value_counts().index
sns.countplot(y=prod, order=order, ax=ax, color="#4C72B0", edgecolor="none")
ax.set_xlabel("Number of entries"); ax.set_ylabel("")
ax.set_title("a) Major product", loc="left", fontsize=9)
for i, v in enumerate(prod.value_counts().reindex(order).values):
    ax.text(v + 0.25, i, str(v), va="center", fontsize=6)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)

# --- b) rate type ---
ax = axes[0, 1]
order = taux.value_counts().index
sns.countplot(y=taux, order=order, ax=ax, color="#55A868", edgecolor="none")
ax.set_xlabel("Number of entries"); ax.set_ylabel("")
ax.set_title("b) Reported rate type", loc="left", fontsize=9)
for i, v in enumerate(taux.value_counts().reindex(order).values):
    ax.text(v + 0.25, i, str(v), va="center", fontsize=6)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)

# --- c) light source ---
ax = axes[1, 0]
order = light.value_counts().index
sns.countplot(y=light, order=order, ax=ax, color="#8172B2", edgecolor="none")
ax.set_xlabel("Number of entries"); ax.set_ylabel("")
ax.set_title("c) Light source", loc="left", fontsize=9)
for i, v in enumerate(light.value_counts().reindex(order).values):
    ax.text(v + 0.25, i, str(v), va="center", fontsize=6)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)

# --- d) co-catalyst vs sacrificial agent (paired bars) ---
ax = axes[1, 1]
cc = cocat.value_counts()
sa = sacr.value_counts()
idx = sorted(set(cc.index) | set(sa.index),
             key=lambda k: -(cc.get(k, 0) + sa.get(k, 0)))
y = np.arange(len(idx))
h = 0.38
ax.barh(y - h/2, [cc.get(k, 0) for k in idx], height=h,
        color="#C44E52", edgecolor="none", label="Co-catalyst")
ax.barh(y + h/2, [sa.get(k, 0) for k in idx], height=h,
        color="#CCB974", edgecolor="none", label="Sacrificial agent")
ax.set_yticks(y); ax.set_yticklabels(idx)
ax.invert_yaxis()
ax.set_xlabel("Number of entries"); ax.set_ylabel("")
ax.set_title("d) Co-catalyst vs sacrificial agent", loc="left", fontsize=9)
ax.legend(frameon=False, loc="lower right", fontsize=7)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)

fig.suptitle(f"Reaction conditions and target (n = {len(df)})",
             x=0.01, ha="left", fontsize=10)
fig.tight_layout(rect=[0, 0, 1, 0.96])
save(fig, "fig03_conditions_target")

saved: fig03_conditions_target
